# ZapLine — removing a line without digging a hole

A notch filter and ZapLine both delete the 50 Hz peak. They differ in what they do to the spectrum *around* it, and in whether they can follow contamination that changes over the recording.

*Deep dive behind the [five-minute demo](../meta_mne_denoise_demo.ipynb).*

## Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("demo_utils.py").exists():
    done = subprocess.run(
        ["git", "clone", "-q", "--depth", "1",
         "https://github.com/snesmaeili/mne-denoise-meta-demo.git"],
        capture_output=True, text=True)
    if done.returncode != 0:
        raise SystemExit(
            "Could not clone the demo repository. If it is still private, the Colab "
            "VM has no credentials for it -- authorising Colab lets it OPEN a "
            "notebook, not clone the repo. Make it public, or run locally."
        )
    os.chdir("mne-denoise-meta-demo")
    sys.path.insert(0, os.getcwd())
try:
    import mne_denoise  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "mne-denoise @ git+https://github.com/mne-tools/mne-denoise.git@f5b821cc2a535e84ed46085d45ea5a356dd8d548"],
                   check=True)

import warnings, logging
import numpy as np
import matplotlib.pyplot as plt
import mne

mne.set_log_level("ERROR")
logging.getLogger("mne_denoise").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*Epochs are not baseline corrected.*")
%matplotlib inline
RANDOM_STATE = 97

## A line that moves

Synthetic 32-channel EEG whose line component changes spatial pattern and amplitude halfway through — the case a fixed filter cannot track.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
sfreq, n_ch, dur = 500.0, 32, 120.0
n = int(sfreq * dur)
t = np.arange(n) / sfreq

# pink-ish background
X = rng.standard_normal((n_ch, n))
X = np.cumsum(X, axis=1) * 0.02 + X
X *= 1e-5 / X.std()

# a 50 Hz component that changes spatial direction and grows at the midpoint
v1 = rng.standard_normal(n_ch); v1 /= np.linalg.norm(v1)
v2 = rng.standard_normal(n_ch); v2 /= np.linalg.norm(v2)
line = np.sin(2 * np.pi * 50.0 * t)
half = n // 2
X[:, :half] += 1.5e-5 * np.outer(v1, line[:half])
X[:, half:] += 4.0e-5 * np.outer(v2, line[half:])

info = mne.create_info([f"EEG{i:03d}" for i in range(n_ch)], sfreq, "eeg")
raw = mne.io.RawArray(X, info, verbose="ERROR")
print(raw)

## Three treatments

In [ ]:
from mne_denoise.zapline import ZapLine
from mne_denoise.qa import noise_surround_ratio, below_noise_distortion_db

notch = raw.copy().notch_filter(freqs=[50.0], verbose="ERROR")

zap_fixed = ZapLine(sfreq=sfreq, line_freq=50.0, n_select="auto")
clean_fixed = zap_fixed.fit_transform(raw.copy())

zap_adapt = ZapLine(sfreq=sfreq, line_freq=50.0, adaptive=True, n_select="auto")
clean_adapt = zap_adapt.fit_transform(raw.copy())

print(f"ZapLine  (fixed)    removed {zap_fixed.n_removed_} components")
print(f"ZapLine+ (adaptive) removed {zap_adapt.adaptive_results_['n_removed']} "
      f"across {len(zap_adapt.adaptive_results_['chunk_info'])} chunks")

## R(50 Hz): how close is the residual to the local floor?

`R = 1` means the residual sits exactly on the surrounding spectral floor. Below 1 means the method removed more than the artifact.

In [ ]:
kw = dict(method="welch", fmin=1.0, fmax=125.0, n_fft=8192, verbose="ERROR")
psds = {}
for name, obj in [("original", raw), ("notch", notch),
                  ("zapline", clean_fixed), ("zapline+", clean_adapt)]:
    p = obj.compute_psd(**kw)
    psds[name] = (p.freqs, p.get_data())

f = psds["original"][0]
print(f"{'':12s} {'R(50)':>8s}  {'distortion 1-45 Hz':>20s}")
for name, (fr, P) in psds.items():
    r = float(np.median(noise_surround_ratio(fr, P, 50.0, peak_bw=0.5)))
    d = "" if name == "original" else (
        f"{float(np.median(below_noise_distortion_db(f, psds['original'][1], P, exclude_freq=50.0))):.4f} dB")
    print(f"{name:12s} {r:8.2f}  {d:>20s}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
band = (f >= 40) & (f <= 65)
for name, colour in [("original", "#333333"), ("notch", "#CC79A7"),
                     ("zapline", "#56B4E9"), ("zapline+", "#E69F00")]:
    ax.semilogy(f[band], psds[name][1].mean(0)[band], label=name, lw=2.2, color=colour)
ax.set_xlabel("Frequency (Hz)"); ax.set_ylabel("PSD (V²/Hz)")
ax.set_title("The notch digs below the floor; ZapLine stops at it")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.show()

## What the fitted object records

In [ ]:
for c in zap_adapt.adaptive_results_["chunk_info"][:8]:
    print(f"  {c['start']/sfreq:6.1f}-{c['end']/sfreq:6.1f}s  "
          f"fine_freq={c['fine_freq']:.2f} Hz  removed={c['n_removed']}  "
          f"artifact_present={bool(c['artifact_present'])}")
print("\nEigenvalues (line-power ratio per component):")
print(np.round(zap_adapt.eigenvalues_[:8], 4))

> **Caveat worth knowing.** `n_select='auto'` can flag many components when the eigenvalue spectrum is flat — i.e. when there is no line artifact to find. `max_prop_remove` (default 0.2) is what bounds it. Check `artifact_present` against `n_removed` on your own data.